In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# from pyspark.sql import functions as F


# from notebookutils import mssparkutils
# print(mssparkutils.lakehouse.get("lh_insurance_bronze_silver"))

from pyspark.sql import functions as F

workspace_id = "3533e4eb-ba19-4350-b072-e6497106fb81"
lakehouse_id = "c5d42bb8-2bbc-4172-a40b-8fbcdee30e29"
base_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files"

files = {
    "claims": f"{base_path}/claims.csv",
    "policies": f"{base_path}/policies.csv",
    "customers": f"{base_path}/customers.csv",
    "vehicles": f"{base_path}/vehicles.csv",
    "payments": f"{base_path}/payments.csv"
}

dfs = {}

for name, path in files.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
        .withColumn("ingestion_ts", F.current_timestamp())
        .withColumn("record_source", F.lit(f"{name}_csv"))
        .withColumn("source_file_name", F.lit(path.split("/")[-1]))
    )
    dfs[name] = df
    print(f"{name}: {df.count()} rows")
    df.show(5, truncate=False)

dfs["claims"].write.mode("overwrite").format("delta").saveAsTable("bronze_claims_raw")
dfs["policies"].write.mode("overwrite").format("delta").saveAsTable("bronze_policies_raw")
dfs["customers"].write.mode("overwrite").format("delta").saveAsTable("bronze_customers_raw")
dfs["vehicles"].write.mode("overwrite").format("delta").saveAsTable("bronze_vehicles_raw")
dfs["payments"].write.mode("overwrite").format("delta").saveAsTable("bronze_payments_raw")

StatementMeta(, f4d88cc3-b013-42f2-bbcd-471f76614f15, 3, Finished, Available, Finished, False)

claims: 12 rows
+------------+-------------+--------+--------------------+-------------+-------------+-------------+---------+------------+--------------+---------------+-------------+--------------------------+-------------+----------------+
|claim_number|policy_number|vin     |claimant_customer_id|incident_date|reported_date|claim_status |loss_type|claim_amount|reserve_amount|fraud_indicator|source_system|ingestion_ts              |record_source|source_file_name|
+------------+-------------+--------+--------------------+-------------+-------------+-------------+---------+------------+--------------+---------------+-------------+--------------------------+-------------+----------------+
|CLM-1001    |POL-2001     |VIN-3001|CUST-4001           |2026-01-04   |2026-01-05   |Open         |Collision|5400.0      |1200.0        |N              |claims_sys   |2026-04-29 01:43:21.262867|claims_csv   |claims.csv      |
|CLM-1002    |POL-2002     |VIN-3002|CUST-4002           |2026-01-07   |2026